# Semantic search with sentence embeddings

The foundation of modern RAG: encode every document into a vector, encode the query into a vector,
return the documents whose vectors are closest to the query vector.

Uses `sentence-transformers/all-MiniLM-L6-v2` — a tiny (~80 MB) but surprisingly good general-purpose
embedding model that produces 384-dimensional vectors.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('embedding dim:', model.get_sentence_embedding_dimension())

In [ ]:
# A tiny docker-themed FAQ corpus.
documents = [
    'A Dockerfile is a text file with instructions for building a container image.',
    'Use multi-stage builds to keep build tools out of the final runtime image.',
    'docker compose lets you define and run multi-container applications.',
    'A volume mount maps a host directory into a container so files persist after the container exits.',
    'A bind mount and a named volume behave differently for permissions and portability.',
    'Layer caching speeds up rebuilds when earlier instructions have not changed.',
    'EXPOSE only documents which ports a container listens on; it does not publish them.',
    'To publish a port to the host, use docker run -p host_port:container_port.',
    'A distroless image contains only your app and its runtime, no shell or package manager.',
    'Running as root inside a container is a security risk — prefer a dedicated user.',
]

doc_embeddings = model.encode(documents, normalize_embeddings=True)
print('embedded', len(documents), 'documents → shape', doc_embeddings.shape)

In [ ]:
def search(query, k=2):
    q = model.encode(query, normalize_embeddings=True)
    # normalized vectors → dot product == cosine similarity
    scores = doc_embeddings @ q
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), documents[i]) for i in top]

for query in [
    'how do I make my docker images smaller?',
    'why is my container not reachable from the host?',
    'is it safe to run as root?',
]:
    print(f'\nQ: {query}')
    for score, doc in search(query):
        print(f'  [Score: {score:.2f}] {doc}')

### Dive deeper

- Add your own documents and queries to see how the model behaves on out-of-domain text.
- Replace the linear scan with a real vector store like `faiss` or `chromadb` and benchmark with a larger corpus.
- Compare against a larger embedding model such as `BAAI/bge-base-en-v1.5` (~440 MB) — worth the image-size tradeoff?